In [2]:
## Definition of the DrugSDA SCP server, including basic operations such as connect, disconnect, list_tools, and parse_result.
import asyncio
import json
from mcp.client.streamable_http import streamablehttp_client
from mcp import ClientSession

DrugSDA_Tool_SERVER_URL = "https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool"      ## DrugSDA-Tool Server
# DrugSDA_Tool_SERVER_URL = "http://180.184.86.2:32208/mcp"

class DrugSDAClient:    
    def __init__(self, server_url: str):
        self.server_url = server_url
        self.session = None
        
    async def connect(self):
        print(f"server url: {self.server_url}")
        try:
            self.transport = streamablehttp_client(
                url=self.server_url,
                headers={"SCP-HUB-API-KEY": "REDACTED_MOLCLAW_KEY"}
            )
            self.read, self.write, self.get_session_id = await self.transport.__aenter__()
            
            self.session_ctx = ClientSession(self.read, self.write)
            self.session = await self.session_ctx.__aenter__()

            await self.session.initialize()
            session_id = self.get_session_id()
            
            print(f"✓ connect success")
            return True
            
        except Exception as e:
            print(f"✗ connect failure: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    async def disconnect(self):
        try:
            if self.session:
                await self.session_ctx.__aexit__(None, None, None)
            if hasattr(self, 'transport'):
                await self.transport.__aexit__(None, None, None)
            print("✓ already disconnect")
        except Exception as e:
            print(f"✗ disconnect error: {e}")
    
    async def list_tools(self):        
        try:
            tools_list = await self.session.list_tools()
            print(f"tool count: {len(tools_list.tools)}")
            
            for i, tool in enumerate(tools_list.tools, 1):
                print(f"{i:2d}. {tool.name}")
                if tool.description:
                    desc_line = tool.description.split('\n')[0]
                    print(f"    {desc_line}")
            
            print(f"✓ Get tool list success")
            return tools_list.tools
            
        except Exception as e:
            print(f"✗ Get tool list fail: {e}")
            return []
    
    def parse_result(self, result):
        try:
            if hasattr(result, 'content') and result.content:
                content = result.content[0]
                if hasattr(content, 'text'):
                    return json.loads(content.text)
            return str(result)
        except Exception as e:
            return {"error": f"parse error: {e}", "raw": str(result)}

In [ ]:
## pred_protein_structure_esmfold
async def main():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    ## Input protein sequence
    sequence = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVVRIELKGIDFKEDGNILGHKLEYNYNSHNVYITADKQKNGIKANFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITLGMDELYK"
    
    for i in range(1):
        result = await client.session.call_tool(
            "pred_protein_structure_esmfold",
            arguments={
                "sequence": sequence
            }
        )
        
        result_data = client.parse_result(result)
        print (result_data)
    
    await client.disconnect()

if __name__ == '__main__':
    await main()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success


In [ ]:
## pred_molecule_admet
async def main():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return

    ## Input SMILES list
    smiles_list = ["N[C@@H](Cc1ccc(O)cc1)C(=O)O", "CC(C)C1=CC=CC=C1"]
    
    for i in range(10):
        result = await client.session.call_tool(
            "pred_mol_admet",
            arguments={
                "smiles_list": smiles_list,
                "smiles_file": ''
            }
        )
        
        result_data = client.parse_result(result)
        print (result_data)
    
    await client.disconnect()

if __name__ == '__main__':
    await main()


In [ ]:
## molecule_docking_quickvina
async def main():
    model_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await model_client.connect():
        print("connection failed")
        return

    ## Input SMILES list
    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/source/exp_data/O76074_1TBF_fix_0209_114217479.pdb"
    smiles_list = ["COc1ccc(-c2cnc(N3Cc4c(nc5ccccc5c4O)C3c3ccc4c(c3)CCO4)nc2)cc1OC", "COc1ccc(-c2cnc(N3Cc4c(nc5ccccc5c4O)C3c3ccc(OC)c(OC)c3)nc2)cc1OC"]
    
    result = await model_client.session.call_tool(
        "molecule_docking_quickvina_fullprocess",
        arguments={
            "pdb_file_path": pdb_file_path,
            "smiles": smiles_list[0],
            "pocket_center_x": 23.1996,
            "pocket_center_y": 39.2113,
            "pocket_center_z": 72.0084,
            "pocket_size_x": 25.0,
            "pocket_size_y": 25.0,
            "pocket_size_z": 25.0
        }
    )
    
    result_data = model_client.parse_result(result)
    print (result_data)
    
    await model_client.disconnect()

if __name__ == '__main__':
    await main()


In [ ]:
## calculate_dleps_score
async def main():
    model_client = DrugSDAClient(DrugSDA_Model_SERVER_URL)
    if not await model_client.connect():
        print("connection failed")
        return

    ## Input SMILES list
    smiles_list = ["Nc1nnc(S(=O)(=O)NCCc2ccc(O)cc2)s1", "COc1ccc2c(=O)cc(C(=O)N3CCN(c4ccc(F)cc4)CC3)oc2c1"]
    
    result = await model_client.session.call_tool(
        "calculate_dleps_score",
        arguments={
            "smiles_list": smiles_list,
            "disease_name": "Aging" 
        }
    )
    
    result_data = model_client.parse_result(result)
    print (result_data)
    
    await model_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## boltz_binding_affinity
DrugSDA_Model_SERVER_URL = "http://180.184.86.2:32208/mcp"

async def main():
    model_client = DrugSDAClient(DrugSDA_Model_SERVER_URL)
    if not await model_client.connect():
        print("connection failed")
        return

    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return
    
    input_pdb_path = "/root/lwj/wll/code/DrugAgentTools/data_fetch/protein_structures/P28221_7E32.pdb"

    ## step 1. pdb fix
    result = await tool_client.session.call_tool(
        "fix_pdb_dock",
        arguments={
            "pdb_file_path": input_pdb_path
        }
    )
    result_data = tool_client.parse_result(result)
    print (result_data)
    fix_pdb_path = result_data['fix_pdb_file_path']

    ## step 2. chains info
    result = await tool_client.session.call_tool(
        "extract_pdb_chains",
        arguments={
            "pdb_file_path": fix_pdb_path
        }
    )
    result_data = tool_client.parse_result(result)
    print ('chain count = ', len(result_data['chains']))
    protein_chains = result_data['chains'][:1]
    print (protein_chains)

    ## step 3. boltz-2
    smiles_list = ['Cc1ccc2c(OCCN3CCN(Cc4ccc(F)c([N+](=O)[O-])c4)CC3)cccc2n1']
    result = await model_client.session.call_tool(
        "boltz_binding_affinity",
        arguments={
            "protein": protein_chains,
            "smiles_list": smiles_list
        }
    )
    
    result_data = model_client.parse_result(result)
    print (result_data)
    
    await tool_client.disconnect()
    await model_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## run_fpocket
async def main():
    model_client = DrugSDAClient(DrugSDA_Model_SERVER_URL)
    if not await model_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv_fix_chainA.pdb"
    
    result = await model_client.session.call_tool(
        "run_fpocket",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )
    
    result_data = model_client.parse_result(result)
    print (result_data)
    
    await model_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## pred_pocket_prank
async def main():
    model_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await model_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv_fix_chainA.pdb"
    
    result = await model_client.session.call_tool(
        "pred_pocket_prank",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )
    
    result_data = model_client.parse_result(result)
    print (result_data)
    
    await model_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## is_valid_protein_sequence
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    test_sequences = [
        "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPA",
        "CC(C)CO"
    ]

    result = await tool_client.session.call_tool(
        "is_valid_protein_sequence",
        arguments={
            "sequences": test_sequences
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## is_valid_smiles
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    ## Input SMILES list
    test_smiles = [
        "CC(C)C",  # 异丁烷
        "",  # 空字符串
        "C1CCCC1",  # 错误：5元环用C1, 但只有一个C1
        "CC(C",  # 错误：括号不匹配
        "CC)",  # 错误：括号不匹配
        "C%10C%11",  # 错误：环闭合数字不匹配
        "C1CC2C1CC2",  # 正确：两个环共享一个键
        "C1CC1C2CC2",  # 正确：两个独立的三元环
        "C1CC1C1CC1",  # 错误：重复使用环数字
        "C1CC1C%10C%10",  # 正确：不同环数字
        "C.CC",  # 正确：两个分子用点分隔
        "InvalidSMILES",  # 无效字符
        "C1CCC",  # 无效：环闭合未完成
    ]

    result = await tool_client.session.call_tool(
        "is_valid_smiles",
        arguments={
            "smiles_list": test_smiles
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## convert_smiles_to_format
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    smiles_list = ["O=C(Nc1cccc2c1CCCC2)N1CCc2c([nH]c3ccccc23)C1c1cccc(F)c1F", "O=C(Nc1cccc(C2CN3CCSC3c3ccccc32)c1)Nc1ccc2ccccc2c1"]

    result = await tool_client.session.call_tool(
        "convert_smiles_to_format",
        arguments={
            "inputs": smiles_list,
            "target_format": "pdbqt"
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## convert_pdb_to_pdbqt_dock
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv_fix_chainA.pdb"

    result = await tool_client.session.call_tool(
        "convert_pdb_to_pdbqt_dock",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)
    
    await tool_client.disconnect()
    
if __name__ == '__main__':
    await main()
    

In [ ]:
## convert_complex_cif_to_pdb
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    cif_file_path = '/root/lwj/wll/code/DrugAgentTools/boltz_data/case_0_model_0.cif'

    result = await tool_client.session.call_tool(
        "convert_complex_cif_to_pdb",
        arguments={
            "cif_file_path": cif_file_path
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## visualize_protein
from IPython.display import Image, display

async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/source/exp_data/5XYF_chainsAC.pdb"

    result = await tool_client.session.call_tool(
        "visualize_protein",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)
    
    display(Image(filename=result_data["image_path"]))

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## visualize_molecule
from IPython.display import Image, display

async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    input_smiles = "O=C(Nc1cccc2c1CCCC2)N1CCc2c([nH]c3ccccc23)C1c1cccc(F)c1F"

    result = await tool_client.session.call_tool(
        "visualize_molecule",
        arguments={
            "input": input_smiles
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)
    
    display(Image(filename=result_data["image_path"]))

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## visualize_complex
from IPython.display import Image, display

async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/case_0_model_0.pdb"

    result = await tool_client.session.call_tool(
        "visualize_complex",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)
    
    display(Image(filename=result_data["global_image_path"]))
    display(Image(filename=result_data["local_image_path"]))

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()
    

In [ ]:
## retrieve_protein_data_by_pdbcode
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_code = '6vkv'

    result = await tool_client.session.call_tool(
        "retrieve_protein_sequence",
        arguments={
            "identifier": "PDE5A",
            "organism": "Homo sapiens"
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## retrieve_smiles_by_compoundname
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    compound_names = ["aspirin", "caffeine", "glucose", "ibuprofen"]

    result = await tool_client.session.call_tool(
        "retrieve_smiles_by_compoundname",
        arguments={
            "compound_names": compound_names
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## fix_pdb_dock
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv.pdb"

    result = await tool_client.session.call_tool(
        "fix_pdb_dock",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## read_smi_file

In [ ]:
## read_fasta_file
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    read_fasta_file = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv.fasta"

    result = await tool_client.session.call_tool(
        "read_fasta_file",
        arguments={
            "read_fasta_file": read_fasta_file
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_mol_basic_info
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    smiles_list = ["O=C(Nc1cccc2c1CCCC2)N1CCc2c([nH]c3ccccc23)C1c1cccc(F)c1F", "O=C(Nc1cccc(C2CN3CCSC3c3ccccc32)c1)Nc1ccc2ccccc2c1"]

    result = await tool_client.session.call_tool(
        "calculate_mol_basic_info",
        arguments={
            "smiles_list": smiles_list
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_mol_hydrophobicity
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    smiles_list = ["O=C(Nc1cccc2c1CCCC2)N1CCc2c([nH]c3ccccc23)C1c1cccc(F)c1F", "O=C(Nc1cccc(C2CN3CCSC3c3ccccc32)c1)Nc1ccc2ccccc2c1"]

    result = await tool_client.session.call_tool(
        "calculate_mol_hydrophobicity",
        arguments={
            "smiles_list": smiles_list
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_mol_hbond
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    smiles_list = ["O=C(Nc1cccc2c1CCCC2)N1CCc2c([nH]c3ccccc23)C1c1cccc(F)c1F", "O=C(Nc1cccc(C2CN3CCSC3c3ccccc32)c1)Nc1ccc2ccccc2c1"]

    result = await tool_client.session.call_tool(
        "calculate_mol_hbond",
        arguments={
            "smiles_list": smiles_list
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_mol_structure_complexity
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    smiles_list = ["O=C(Nc1cccc2c1CCCC2)N1CCc2c([nH]c3ccccc23)C1c1cccc(F)c1F", "O=C(Nc1cccc(C2CN3CCSC3c3ccccc32)c1)Nc1ccc2ccccc2c1"]

    result = await tool_client.session.call_tool(
        "calculate_mol_structure_complexity",
        arguments={
            "smiles_list": smiles_list
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_mol_topology
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    smiles_list = ["O=C(Nc1cccc2c1CCCC2)N1CCc2c([nH]c3ccccc23)C1c1cccc(F)c1F", "O=C(Nc1cccc(C2CN3CCSC3c3ccccc32)c1)Nc1ccc2ccccc2c1"]

    result = await tool_client.session.call_tool(
        "calculate_mol_topology",
        arguments={
            "smiles_list": smiles_list
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_mol_drug_chemistry
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    smiles_list = ["O=C(Nc1cccc2c1CCCC2)N1CCc2c([nH]c3ccccc23)C1c1cccc(F)c1F", "O=C(Nc1cccc(C2CN3CCSC3c3ccccc32)c1)Nc1ccc2ccccc2c1"]

    result = await tool_client.session.call_tool(
        "calculate_mol_drug_chemistry",
        arguments={
            "smiles_list": smiles_list
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_mol_charge
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    smiles_list = ["O=C(Nc1cccc2c1CCCC2)N1CCc2c([nH]c3ccccc23)C1c1cccc(F)c1F", "O=C(Nc1cccc(C2CN3CCSC3c3ccccc32)c1)Nc1ccc2ccccc2c1"]

    result = await tool_client.session.call_tool(
        "calculate_mol_charge",
        arguments={
            "smiles_list": smiles_list
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_mol_complexity
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    smiles_list = ["O=C(Nc1cccc2c1CCCC2)N1CCc2c([nH]c3ccccc23)C1c1cccc(F)c1F", "O=C(Nc1cccc(C2CN3CCSC3c3ccccc32)c1)Nc1ccc2ccccc2c1"]

    result = await tool_client.session.call_tool(
        "calculate_mol_complexity",
        arguments={
            "smiles_list": smiles_list
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_protein_sequence_properties
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    sequence = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVVRIELKGIDFKEDGNILGHKLEYNYNSHNVYITADKQKNGIKANFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITLGMDELYK"

    result = await tool_client.session.call_tool(
        "calculate_protein_sequence_properties",
        arguments={
            "sequence": sequence
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_pdb_basic_info
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv_fix_chainA.pdb"

    result = await tool_client.session.call_tool(
        "calculate_pdb_basic_info",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_pdb_structural_geometry
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv_fix_chainA.pdb"

    result = await tool_client.session.call_tool(
        "calculate_pdb_structural_geometry",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_pdb_quality_metrics
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv_fix_chainA.pdb"

    result = await tool_client.session.call_tool(
        "calculate_pdb_quality_metrics",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_pdb_composition_info
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv_fix_chainA.pdb"

    result = await tool_client.session.call_tool(
        "calculate_pdb_composition_info",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## calculate_smiles_similarity
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    target = "CCO"  # 乙醇
    candidates = [
        "CCCO",      # 丙醇
        "CCCCO",     # 丁醇
        "CC(C)O",    # 异丙醇
        "CCC(C)O",   # 仲丁醇
        "C1CC1",     # 环丙烷
        "CC=O",      # 乙醛
        "CCCOO"      # 丙酸
    ]

    result = await tool_client.session.call_tool(
        "calculate_common_fragments",
        arguments={
            "target_smiles": target,
            "candidate_smiles_list": candidates,
            "radius": 2
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## extract_and_save_chains
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv_fix.pdb"

    result = await tool_client.session.call_tool(
        "extract_and_save_chains",
        arguments={
            "pdb_file_path": pdb_file_path,
            "chain_ids": ["A", "B"]
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## extract_pdb_chains
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    pdb_file_path = "/root/lwj/wll/code/DrugAgentTools/exp_data/6vkv.pdb"

    result = await tool_client.session.call_tool(
        "extract_pdb_chains",
        arguments={
            "pdb_file_path": pdb_file_path
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## search_uniprot_id
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    result = await tool_client.session.call_tool(
        "search_uniprot_id",
        arguments={
            "gene_name": "PDE5A",
            "organism": "9606"
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## analyze_protein_ligand_interactions
async def main():
    tool_client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await tool_client.connect():
        print("connection failed")
        return

    result = await tool_client.session.call_tool(
        "analyze_protein_ligand_interactions",
        arguments={
            "complex_file": "/root/lwj/wll/code/DrugAgentTools/source/exp_data/case_0_model_0_0313_174841337.pdb",
            "ligand_identifier": "auto"
        }
    )

    result_data = tool_client.parse_result(result)
    print (result_data)

    await tool_client.disconnect()

if __name__ == '__main__':
    await main()

In [ ]:
## reinvent_mol2mol_sampling